In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# Loading the song information
music_info = pd.read_csv("../data/music_info_clean.csv")

# Loading the interaction matrix
interaction_matrix = load_npz("../data/interaction_matrix.npz")

# Lloading the user and trackID's
user_ids = np.load("../user_ids.npy", allow_pickle=True)
track_ids = np.load("../track_ids.npy", allow_pickle=True)

print("Everything loaded succesfully")


# Checks to make sure the mappigs match the matrix
print("Number of user IDs:", len(user_ids))
print("Matrix rows:", interaction_matrix.shape[0])

print("Number of track IDs:", len(track_ids))
print("Matrix columns:", interaction_matrix.shape[1])


Everything loaded succesfully
Number of user IDs: 290898
Matrix rows: 290898
Number of track IDs: 29922
Matrix columns: 29922


In [8]:
# Creating dictionaries that connect the actual user/track IDs to the positions in the matrix
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
track_to_idx = {track_id: idx for idx, track_id in enumerate(track_ids)}

idx_to_user = {idx: user_id for idx, user_id in enumerate(user_ids)}
idx_to_track = {idx: track_id for idx, track_id in enumerate(track_ids)}

In [15]:
# Creating the cosine similarity function
def find_similar_users(user_id,top_n=5 ):

    # Checking if the user exists
    if user_id not in user_to_idx:
        print("User not found")
        return[]

    # Getting row index of the target user
    user_index = user_to_idx[user_id]

    # Getting the targets listening data
    target_user = interaction_matrix[user_index]

    # Calculating the cosine similarity
    similarities = cosine_similarity(target_user, interaction_matrix).flatten()

    # Excluding the target user
    similarities[user_index] = -1

    # Finding similar users idices 
    similar_indices = similarities.argsort()[::-1][:top_n]

    # storing similar users and their similarity scores
    similar_users=[]

    for index in similar_indices:
        similar_users.append((idx_to_user[index], similarities[index]))

    return similar_users

In [46]:
# Testing the cosine similarity function on the first user
target_user = user_ids[100]

# Find the 5 users with the most similar listening patterns
similar_users = find_similar_users(
    target_user,
    top_n=5
)

# Display the target user and their most similar users
print("Target User:", target_user)

for user, similarity in similar_users:
    print(user, similarity)

Target User: 6beb4699102775dab57aa406c5ea1217c4ff4869
0a964e1e55b0f6bc2049c7e6bc2351e682433945 0.5611127092082491
b2fc3f38684b6e5815000df4d18deb6b351b1a3f 0.5610468789121451
920b76ef86e87464cf8012fa7910e0734d7a3ff3 0.5456463884154851
8cf36d8170bc5422a074f5948e52af21719c8c28 0.5389831358825794
185c9b64a10479a5cf7e6a6c4286144e82abba7a 0.5314981395767449


In [48]:
## Function that is uses similar users to score songs the target user has not listened to
def recommend_songs(user_id, top_users=5, top_songs=5):

    # Checking if the user exists
    if user_id not in user_to_idx:
        print("User not found")
        return []

    # Getting similar users
    similar_users = find_similar_users(user_id, top_n = top_users)

    # Getting the target user's row
    user_index = user_to_idx[user_id]
    target_vector = interaction_matrix[user_index]

    # Songs that the target user has listened to already
    listened_tracks = set(target_vector.indices)

    # Storing the recommended songs
    recommendation_scores = {}

    # Going through each similar user
    for similar_user_id, similarity_score in similar_users:
        similar_user_index = user_to_idx[similar_user_id]
        similar_user_vector = interaction_matrix[similar_user_index]

        # Checking the song each similar user listened to
        for track_index, playcount in zip(similar_user_vector.indices, similar_user_vector.data):

            # Skipping songs the target user has already listened to
            if track_index in listened_tracks:
                continue

            # Weighted recommended songs
            score = similarity_score * playcount

            if track_index in recommendation_scores:
                recommendation_scores[track_index] += score
            else:
                recommendation_scores[track_index] = score


    # Sorting the songs from highest similarity score to lowest
    sorted_tracks = sorted(recommendation_scores.items(), key=lambda x: x[1], reverse = True)


    # Keeping on the top songs
    top_tracks = sorted_tracks[:top_songs]
    return top_tracks

In [49]:
# Function to get the song details
def recommended_song_details(recommendations):
    results = []
    for track_index, score in recommendations:

        # Converting the matrix column index to the actual Track ID
        track_id = idx_to_track[int(track_index)]

        # Finding the track in the dataset
        song = music_info[music_info["track_id"] == track_id]

        # Only adds if the track exist in the music_info
        if not song.empty:
            song_name = song.iloc[0]["name"]
            artist = song.iloc[0]["artist"]

            results.append({"track_id": track_id, "song": song_name, "artist": artist, "score": float(score)})

            return results
                           

In [50]:
# Combines the two steps into one clean function that return readable

def recommend_for_user(user_id, top_users=5, top_songs=5):

    # Ask for more candidates than we actually need
    recommendations = recommend_songs(
        user_id,
        top_users=top_users,
        top_songs=top_songs * 10
    )

    song_details = []

    for track_index, score in recommendations:

        track_id = idx_to_track[int(track_index)]

        song = music_info[
            music_info["track_id"] == track_id
        ]

        if not song.empty:

            song_details.append(
                {
                    "track_id": track_id,
                    "song": song.iloc[0]["name"],
                    "artist": song.iloc[0]["artist"],
                    "score": float(score)
                }
            )

        # Stop once we have enough valid songs
        if len(song_details) == top_songs:
            break

    return song_details

In [51]:
final_recommendations = recommend_for_user(
    target_user,
    top_users=5,
    top_songs=5
)

print("Recommended Songs:\n")

for i, song in enumerate(final_recommendations, start=1):
    print(f"{i}. {song['song']} - {song['artist']}")

Recommended Songs:

1. Gimme Sympathy - Metric
2. Cover My Eyes - La Roux
3. Thieves In The Night - Black Star
4. Eriatarka - The Mars Volta
5. Where Is My Mind? - Pixies
